In [ ]:
!wget https://github.com/makelove/OpenCV-Python-Tutorial/raw/master/data/slow.flv
!wget https://moderncomputervision.s3.eu-west-2.amazonaws.com/haarcascades.zip
!wget https://github.com/rajeevratan84/ModernComputerVision/raw/main/walking_short_clip.mp4
!wget https://github.com/rajeevratan84/ModernComputerVision/raw/main/walking.avi
!unzip -qq haarcascades.zip

In [2]:
import numpy as np
from matplotlib import pyplot as plt
import cv2
from google.colab.patches import cv2_imshow
from IPython.display import HTML
from base64 import b64encode

In [3]:
def imshow(title="Image", image=None, size=6):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w / h
    plt.figure(figsize=(size * aspect_ratio, size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

# 1) Optical Flow

## 1.1 Lucas-Kanade Optical Flow

Optical flow is the pattern of apparent motion of image objects between two consecutive frames caused by the movemement of object or camera. It is 2D vector field where each vector is a displacement vector showing the movement of points from first frame to second. Consider the image below

![](https://upload.wikimedia.org/wikipedia/commons/thumb/1/10/Optical_flow_example_v2.png/440px-Optical_flow_example_v2.png)

It shows a ball moving in 5 consecutive frames. The arrow shows its displacement vector. Optical flow has many applications in areas like :

- Structure from Motion
- Video Compression
- Video Stabilization

Optical flow works on several assumptions:

- The pixel intensities of an object do not change between consecutive frames.
- Neighbouring pixels have similar motion.

Learn more - https://opencv-python-tutroals.readthedocs.io/en/latest/py_tutorials/py_video/py_lucas_kanade/py_lucas_kanade.html

In [63]:
cap = cv2.VideoCapture("walking.avi")
width = int(cap.get(3))
height = int(cap.get(4))
out = cv2.VideoWriter(
    "optical_flow_walking.avi",
    cv2.VideoWriter_fourcc("M", "J", "P", "G"),
    30,
    (width, height),
)

# Set parameters for ShiTomasi corner detection
# to detect the point we want to track
feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

# parameters for lucas kanade optical flow
# small window more sensitve but might miss larger motion/moving fast
# high level lower resolution
# iterations = 10
lucas_kanade_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
)

# Create some random colors
# Used to create our trails for object movement in the image
color = np.random.randint(0, 255, (100, 3))

# Take first frame and find corners in it
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# Find inital corner locations by Grabbing the corners
prev_corners = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

# Create a mask image for drawing purposes
mask = np.zeros_like(prev_frame)

while 1:
    ret, frame = cap.read()
    if ret == True:
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # calculate optical flow
        new_corners, status, errors = cv2.calcOpticalFlowPyrLK(
            prev_gray, frame_gray, prev_corners, None, **lucas_kanade_params
        )

        # Select and store good points
        good_new = new_corners[status == 1]
        good_old = prev_corners[status == 1]

        # Draw the tracks
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = map(int, new.ravel())
            c, d = map(int, old.ravel())
            mask = cv2.line(mask, (a, b), (c, d), color[i].tolist(), 2)
            frame = cv2.circle(frame, (a, b), 5, color[i].tolist(), -1)

        img = cv2.add(frame, mask)

        # Save Video
        out.write(img)
        # imshow('Optical Flow - Lucas-Kanade',img)

        # Now update the previous frame and previous points
        prev_gray = frame_gray.copy()
        prev_corners = good_new.reshape(-1, 1, 2)

    else:
        break

cap.release()
out.release()

In [ ]:
!ffmpeg -i /content/optical_flow_walking.avi optical_flow_walking.mp4 -y

In [67]:
mp4 = open("optical_flow_walking.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

Output hidden; open in https://colab.research.google.com to view.

## 1.1 Dense Optical Flow in OpenCV

calcOpticalFlowFarneback(prev, next, flow, pyr_scale, levels, winsize, iterations, poly_n, poly_sigma, flags) -> flow

This function computes a dense optical flow using the Gunnar Farneback's algorithm.

Here are the parameters for the function and what they represent:

* prev first 8-bit single-channel input image.
* next second input image of the same size and the same type as prev.
* flow computed flow image that has the same size as prev and type CV_32FC2.
* pyr_scale parameter, specifying the image scale (\<1) to build pyramids for each image
    * pyr_scale=0.5 means a classical pyramid, where each next layer is twice smaller than the previous one.
    
* levels number of pyramid layers including the initial image; levels=1 means that no extra layers are created and only the original images are used.
* winsize averaging window size
    * larger values increase the algorithm robustness to image
* noise and give more chances for fast motion detection, but yield more blurred motion field.
* iterations number of iterations the algorithm does at each pyramid level.
* poly_n size of the pixel neighborhood used to find polynomial expansion in each pixel
    * larger values mean that the image will be approximated with smoother surfaces, yielding more robust algorithm and more blurred motion field, typically poly_n =5 or 7.
* poly_sigma standard deviation of the Gaussian that is used to smooth derivatives used as a basis for the polynomial expansion; for poly_n=5, you can set poly_sigma=1.1, for poly_n=7, a good value would be poly_sigma=1.5.
   

In [4]:
cap = cv2.VideoCapture("walking.avi")
width = int(cap.get(3))
height = int(cap.get(4))
out = cv2.VideoWriter(
    "dense_optical_flow_walking.avi",
    cv2.VideoWriter_fourcc("M", "J", "P", "G"),
    30,
    (width, height),
)

# Get first frame
ret, first_frame = cap.read()
previous_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
hsv = np.zeros_like(first_frame)
hsv[..., 1] = 255

while True:
    ret, frame2 = cap.read()

    if ret == True:
        next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        # Computes the dense optical flow using the Gunnar Farneback’s algorithm
        flow = cv2.calcOpticalFlowFarneback(
            previous_gray, next, None, 0.5, 3, 15, 3, 5, 1.2, 0
        )

        # use flow to calculate the magnitude (speed) and angle of motion
        # use these values to calculate the color to reflect speed and angle
        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        hsv[..., 0] = angle * (180 / (np.pi / 2))
        hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
        final = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # Save Video
        out.write(final)
        previous_gray = next

    else:
        break

cap.release()
out.release()

In [ ]:
!ffmpeg -i /content/dense_optical_flow_walking.avi dense_optical_flow_walking.mp4 -y

In [6]:
mp4 = open("dense_optical_flow_walking.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

Output hidden; open in https://colab.research.google.com to view.

# 2) Mean Shift Tracking
![](https://opencv-python-tutroals.readthedocs.io/en/latest/_images/meanshift_basics.jpg)
Consider you have a set of points. (It can be a pixel distribution like histogram backprojection). You are given a small window ( may be a circle) and you have to move that window to the area of maximum pixel density (or maximum number of points). 
![](https://opencv-python-tutroals.readthedocs.io/en/latest/_images/meanshift_face.gif)

Mean shift is a hill climbing algorithm which involves shifting this kernel iteratively to a higher density region until convergence. Every shift is defined by a mean shift vector. The mean shift vector always points toward the direction of the maximum increase in the density. 
![](https://upload.wikimedia.org/wikipedia/commons/b/bd/Meanshiftred.gif)

Read Paper Here - https://ieeexplore.ieee.org/document/732882

Animation Source - https://fr.wikipedia.org/wiki/Camshift

In [54]:
# Capture a video stream
cap = cv2.VideoCapture("slow.flv")
ret, frame = cap.read()
width = int(cap.get(3))
height = int(cap.get(4))
car_detector = cv2.CascadeClassifier("Haarcascades/haarcascade_car.xml")

# detect the first frame
car_rects = car_detector.detectMultiScale(frame)

# video writter for saved output
out = cv2.VideoWriter(
    "car_tracking_mean_shift.avi",
    cv2.VideoWriter_fourcc("M", "J", "P", "G"),
    30,
    (width, height),
)

# set up the initial location of window for tracking
car_x, car_y, w, h = tuple(car_rects[-1])
track_window = (car_x, car_y, w, h)
roi = frame[car_y : car_y + h, car_x : car_x + w]
# imshow('initial window', roi)

# Use the HSV Color Mapping
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# Find histogram to backproject the target on each frame for calculation of meanshit
mask = cv2.inRange(
    hsv_roi, np.array((0.0, 60.0, 32.0)), np.array((180.0, 255.0, 255.0))
)
roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])

# Normalize the histogram array values given a min of 0 and max of 255
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by at least 1 pt
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

while True:
    ret, frame = cap.read()
    if ret == True:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Calculate the Back Projection based off the roi_hist created earlier
        dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

        # Apply meanshift to get the new coordinates of the rectangle
        ret, track_window = cv2.meanShift(dst, track_window, term_crit)

        # Draw the new rectangle on the image
        x, y, w, h = track_window
        img2 = cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 5)
        out.write(img2)
        # imshow('Tracking', img2)

    else:
        break

cap.release()
out.release()

In [ ]:
!ffmpeg -i /content/car_tracking_mean_shift.avi car_tracking_mean_shift.mp4 -y

In [53]:
mp4 = open("car_tracking_mean_shift.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

# 3) CamShift Tracking

Almost same as meanshift, but it returns a rotated rectangle (that is our result) and box parameters (used to be passed as search window in next iteration). 

![](https://upload.wikimedia.org/wikipedia/commons/8/86/CamshiftStillImage.gif)

Read Paper Here - https://ieeexplore.ieee.org/document/732882

Animation Source - https://fr.wikipedia.org/wiki/Camshift

In [56]:
# Capture a video stream
cap = cv2.VideoCapture("slow.flv")
ret, frame = cap.read()
width = int(cap.get(3))
height = int(cap.get(4))
car_detector = cv2.CascadeClassifier("Haarcascades/haarcascade_car.xml")

# detect the first frame
car_rects = car_detector.detectMultiScale(frame)

# video writter for saved output
out = cv2.VideoWriter(
    "car_tracking_cam_shift.avi",
    cv2.VideoWriter_fourcc("M", "J", "P", "G"),
    30,
    (width, height),
)

# set up the initial location of window for tracking
car_x, car_y, w, h = tuple(car_rects[-1])
track_window = (car_x, car_y, w, h)
roi = frame[car_y : car_y + h, car_x : car_x + w]
# imshow('initial window', roi)

# Use the HSV Color Mapping
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# Find histogram to backproject the target on each frame for calculation of meanshit
# mask = cv2.inRange(hsv_roi, np.array((0., 60., 32.)), np.array((180., 255., 255.)))
roi_hist = cv2.calcHist([hsv_roi], [0], None, [180], [0, 180])

# Normalize the histogram array values given a min of 0 and max of 255
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by at least 1 pt
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

while True:
    ret, frame = cap.read()
    if ret == True:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Calculate the Back Projection based off the roi_hist created earlier
        dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

        # Apply meanshift to get the new coordinates of the rectangle
        ret, track_window = cv2.CamShift(dst, track_window, term_crit)

        # use polylines instead as rectangle as it is not 90 degree
        pts = cv2.boxPoints(ret)
        pts = np.int0(pts)
        img2 = cv2.polylines(frame, [pts], True, 255, 2)
        out.write(img2)
        # imshow('Tracking', img2)

    else:
        break

cap.release()
out.release()

In [ ]:
!ffmpeg -i /content/car_tracking_cam_shift.avi car_tracking_cam_shift.mp4 -y

In [58]:
mp4 = open("car_tracking_cam_shift.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

Output hidden; open in https://colab.research.google.com to view.